# 🚀 Azure Transaction Analytics Platform

**Developer:** Sangam Srivastav  
**Internship:** Celebal Technologies — Data Engineering  
**Tech Stack:** Python · PySpark · Azure Databricks · ADLS Gen2 · Delta Lake · SQL

---

## 📌 Project Overview

This notebook demonstrates a production-grade ETL pipeline that:
1. Ingests multi-channel transaction data (Web, Mobile, In-Store) from Azure Data Lake Storage Gen2
2. Applies enterprise-grade data cleaning and quality validation
3. Computes business KPIs — Customer LTV, Product Performance, Campaign ROI
4. Persists results as Delta Lake tables with ACID guarantees

```
┌──────────────┐    ┌──────────────┐    ┌──────────────────────┐    ┌──────────────┐
│ Data Sources │───▶│  ADLS Gen2   │───▶│   Azure Databricks   │───▶│ Delta Tables │
│ • Web        │    │ • Raw CSV    │    │ • Data Ingestion     │    │ • Analytics  │
│ • Mobile     │    │ • Staging    │    │ • Data Cleaning      │    │ • Insights   │
│ • In-Store   │    │ • Archive    │    │ • Analytics Engine   │    │ • Reports    │
└──────────────┘    └──────────────┘    └──────────────────────┘    └──────────────┘
```

---
## 📦 Step 0 — Install Dependencies & Initialize Spark

In [ ]:
# Install required packages (run this only once)
# In Databricks this is pre-installed — for local testing:
# !pip install pyspark delta-spark matplotlib pandas

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from pyspark.sql.functions import (
    col, count, sum, countDistinct, mean, stddev,
    when, current_timestamp, hash, lit
)
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 6)

print('✅ All imports successful')

In [ ]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName('AzureTransactionAnalytics') \
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension') \
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'✅ Spark version: {spark.version}')
print(f'✅ Spark session initialized: {spark.sparkContext.appName}')

---
## ⚙️ Step 1 — Configuration (config.py)

Sets up the Azure Data Lake Storage Gen2 connection using Databricks secret scope for secure credential management.

In [ ]:
# ============================================================
# config.py — Azure ADLS Gen2 Connection Setup
# ============================================================

def setup_adls_connection():
    """Set up ADLS Gen2 connection using Databricks secret scope."""
    try:
        storage_account = 'mydatalake2004'
        container = 'transaction-data'
        # Secure credential fetch from Databricks secret scope
        access_key = dbutils.secrets.get(scope='azure-storage', key='storage-access-key')
        spark = SparkSession.getActiveSession()
        spark.conf.set(
            f'fs.azure.account.key.{storage_account}.dfs.core.windows.net',
            access_key
        )
        print('✅ ADLS Gen2 connection successful.')
        print(f'   Storage Account : {storage_account}')
        print(f'   Container       : {container}')
        return storage_account, container
    except Exception as e:
        print(f'❌ Failed to connect to ADLS: {e}')
        return None, None

def verify_files_exist(account, container):
    """Verify files exist in ADLS Gen2 container."""
    try:
        base_path = f'abfss://{container}@{account}.dfs.core.windows.net/'
        files = dbutils.fs.ls(base_path)
        print(f'\n📂 Files in container [{container}]:')
        for f in files:
            print(f'   📄 {f.name} — {f.size:,} bytes')
        return [f.name for f in files]
    except Exception as e:
        print(f'❌ Could not list files: {e}')
        return []

# Run connection
account, container = setup_adls_connection()
if account:
    verify_files_exist(account, container)

---
## 📥 Step 2 — Data Ingestion (data_loader.py)

Loads transaction and product CSVs from ADLS Gen2 with **enforced schemas** to prevent type inference errors in production.

In [ ]:
# ============================================================
# data_loader.py — Schema-enforced CSV Ingestion
# ============================================================

# Transaction schema — enforced to prevent type mismatch at scale
transaction_schema = StructType([
    StructField('transaction_id',   StringType(),    False),
    StructField('customer_id',      StringType(),    True),
    StructField('product_id',       StringType(),    False),
    StructField('quantity',         IntegerType(),   True),
    StructField('price',            DoubleType(),    True),
    StructField('transaction_date', TimestampType(), True),
    StructField('campaign_id',      StringType(),    True)
])

# Product schema
product_schema = StructType([
    StructField('product_id',   StringType(), False),
    StructField('description',  StringType(), True),
    StructField('category',     StringType(), True),
    StructField('unit_price',   DoubleType(), True)
])

def load_csv(file_path, file_name, schema=None):
    """Load CSV from ADLS Gen2 with optional schema enforcement."""
    try:
        print(f'📥 Loading {file_name}...')
        spark = SparkSession.getActiveSession()
        reader = spark.read.option('header', 'true')
        df = reader.schema(schema).csv(file_path) if schema else reader.option('inferSchema', 'true').csv(file_path)
        print(f'   ✅ {df.count():,} rows | {len(df.columns)} columns loaded')
        df.show(5, truncate=False)
        return df
    except Exception as e:
        print(f'   ❌ Error loading {file_name}: {e}')
        return None

def load_transaction_data(account, container):
    """Load all transaction and product data from ADLS Gen2."""
    path = f'abfss://{container}@{account}.dfs.core.windows.net/'
    print('=' * 55)
    print('📦 LOADING DATA FROM ADLS Gen2')
    print('=' * 55)
    tx_df = load_csv(path + 'transactions/*.csv', 'transactions', transaction_schema)
    pr_df = load_csv(path + 'products.csv', 'products', product_schema)
    return tx_df, pr_df

# Run data loading
tx_df, pr_df = load_transaction_data(account, container)

---
## 🧹 Step 3 — Data Cleaning & Quality Validation (data_cleaner.py)

Applies:
- **Null removal** — drops rows where all values are null
- **Deduplication** — removes exact duplicate records
- **Outlier detection** — flags anomalies using mean ± 3×stddev (3-sigma rule)
- **Quality flags** — adds quality score, processing timestamp, and record hash

In [ ]:
# ============================================================
# data_cleaner.py — Data Quality & Validation
# ============================================================

def detect_outliers(df, column):
    """Flag outliers using 3-sigma (mean ± 3*stddev) rule."""
    stats = df.select(
        mean(column).alias('mean'),
        stddev(column).alias('std')
    ).collect()[0]
    mean_val, std_val = stats['mean'], stats['std']
    print(f'   📊 {column}: mean={mean_val:.2f}, std={std_val:.2f}')
    print(f'      Outlier range: < {mean_val - 3*std_val:.2f} or > {mean_val + 3*std_val:.2f}')
    return df.withColumn(
        f'{column}_is_outlier',
        (col(column) > (mean_val + 3 * std_val)) | (col(column) < (mean_val - 3 * std_val))
    )

def clean_data(df, name):
    """Clean DataFrame — remove nulls, duplicates, and detect outliers."""
    try:
        print(f'\n🧹 Cleaning [{name}] dataset...')
        before = df.count()
        df = df.dropna(how='all').dropDuplicates()
        after = df.count()
        print(f'   Rows before: {before:,} | After: {after:,} | Removed: {before - after:,}')
        
        # Null value report
        print(f'   Null values per column:')
        df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()
        
        # Outlier detection for numeric transaction fields
        if name == 'transactions':
            print('   🔍 Outlier Detection:')
            df = detect_outliers(df, 'quantity')
            df = detect_outliers(df, 'price')
            outlier_count = df.filter(col('quantity_is_outlier') | col('price_is_outlier')).count()
            print(f'   ⚠️  Total outlier records flagged: {outlier_count:,}')
        return df
    except Exception as e:
        print(f'❌ Error cleaning {name}: {e}')
        return df

def add_quality_flags(df):
    """Add quality score, processing timestamp, and record hash."""
    return df \
        .withColumn('data_quality_score', when(col('transaction_id').isNull(), 0.0).otherwise(1.0)) \
        .withColumn('processed_at', current_timestamp()) \
        .withColumn('record_hash', hash(*[col(c) for c in df.columns]))

# Run cleaning
tx_df  = clean_data(tx_df, 'transactions')
pr_df  = clean_data(pr_df, 'products')
tx_df  = add_quality_flags(tx_df)
pr_df  = add_quality_flags(pr_df)

print('\n✅ Cleaned transaction sample:')
tx_df.select('transaction_id','customer_id','quantity','price','quantity_is_outlier','price_is_outlier','data_quality_score').show(5)

---
## 📊 Step 4 — Channel Enrichment & Analytics Engine (analytics.py)

Enriches data with channel tags (Web/Mobile/In-Store) and computes:
- **Customer Analytics** — LTV, avg order value, outlier flagging
- **Product Analytics** — Units sold, revenue per product
- **Category Analytics** — Cross-category performance
- **Campaign Analytics** — ROI, unique customers, revenue attribution

In [ ]:
# ============================================================
# analytics.py — Business Intelligence Engine
# ============================================================

def enrich_channels(df):
    """Tag transactions with channel based on transaction_id prefix."""
    return df.withColumn('channel',
        when(col('transaction_id').rlike('^WEB'), 'web')
       .when(col('transaction_id').rlike('^MOB'), 'mobile')
       .otherwise('in-store')
    )

def generate_analytics(transactions, products):
    """Join transactions with products and compute all KPIs."""
    print('🔗 Joining transactions with product catalog...')
    joined = transactions.join(products, 'product_id', 'left').cache()
    print(f'   Joined dataset: {joined.count():,} records')

    # Customer-level KPIs
    customer_df = joined.groupBy('customer_id').agg(
        count('transaction_id').alias('total_transactions'),
        sum(col('quantity') * col('price')).alias('total_revenue')
    ).withColumn('avg_order_value', col('total_revenue') / col('total_transactions'))

    # Outlier detection on customer revenue (fraud detection)
    rev_stats = customer_df.select(
        mean('total_revenue').alias('mean'),
        stddev('total_revenue').alias('std')
    ).collect()[0]
    m, s = rev_stats['mean'], rev_stats['std']
    customer_df = customer_df.withColumn(
        'is_revenue_outlier',
        (col('total_revenue') > (m + 3*s)) | (col('total_revenue') < (m - 3*s))
    )

    # Product-level KPIs
    product_df = joined.groupBy('product_id', 'description').agg(
        sum('quantity').alias('units_sold'),
        sum(col('quantity') * col('price')).alias('product_revenue')
    )

    # Category-level KPIs
    category_df = joined.groupBy('category').agg(
        sum('quantity').alias('units_sold'),
        sum(col('quantity') * col('price')).alias('category_revenue')
    )

    # Campaign ROI KPIs
    campaign_df = joined.groupBy('campaign_id').agg(
        count('*').alias('total_transactions'),
        sum(col('quantity') * col('price')).alias('campaign_revenue'),
        countDistinct('customer_id').alias('unique_customers')
    ).withColumn(
        'avg_revenue_per_customer',
        col('campaign_revenue') / col('unique_customers')
    )

    return joined, customer_df, product_df, category_df, campaign_df

# Enrich and generate analytics
tx_df = enrich_channels(tx_df)

print('\n📊 Channel distribution:')
tx_df.groupBy('channel').count().orderBy('count', ascending=False).show()

joined, cust_df, prod_df, cat_df, camp_df = generate_analytics(tx_df, pr_df)

print('\n🏆 Top 5 Customers by Revenue:')
cust_df.orderBy('total_revenue', ascending=False).show(5)

print('\n📦 Top 5 Products by Units Sold:')
prod_df.orderBy('units_sold', ascending=False).show(5)

print('\n🏷️  Category Performance:')
cat_df.orderBy('category_revenue', ascending=False).show()

print('\n📣 Campaign ROI Summary:')
camp_df.orderBy('campaign_revenue', ascending=False).show()

---
## 📉 Step 5 — Visualizations (visualizations.py)

Business charts for stakeholder reporting.

In [ ]:
# ============================================================
# visualizations.py — Business Charts
# ============================================================

def plot_top_customers(customer_df, top_n=10):
    """Bar chart — Top customers by average order value."""
    pdf = customer_df.orderBy('avg_order_value', ascending=False).limit(top_n).toPandas()
    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(pdf['customer_id'], pdf['avg_order_value'], color='#1f77b4', edgecolor='white')
    ax.set_title(f'Top {top_n} Customers by Average Order Value', fontsize=14, fontweight='bold')
    ax.set_xlabel('Customer ID')
    ax.set_ylabel('Avg Order Value ($)')
    ax.bar_label(bars, fmt='$%.0f', padding=3, fontsize=8)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('top_customers.png', dpi=150)
    plt.show()
    print('✅ Chart saved: top_customers.png')

def plot_top_products(product_df, top_n=10):
    """Horizontal bar chart — Top products by units sold."""
    pdf = product_df.orderBy('units_sold', ascending=False).limit(top_n).toPandas()
    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.barh(pdf['description'], pdf['units_sold'], color='#ff7f0e', edgecolor='white')
    ax.set_title(f'Top {top_n} Products by Units Sold', fontsize=14, fontweight='bold')
    ax.set_xlabel('Units Sold')
    ax.bar_label(bars, padding=3, fontsize=8)
    plt.tight_layout()
    plt.savefig('top_products.png', dpi=150)
    plt.show()
    print('✅ Chart saved: top_products.png')

def plot_campaign_revenue(campaign_df):
    """Pie chart — Campaign revenue distribution."""
    pdf = campaign_df.select('campaign_id', 'campaign_revenue').toPandas()
    colors = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b']
    fig, ax = plt.subplots(figsize=(8, 8))
    wedges, texts, autotexts = ax.pie(
        pdf['campaign_revenue'], labels=pdf['campaign_id'],
        autopct='%1.1f%%', colors=colors[:len(pdf)], startangle=140
    )
    ax.set_title('Campaign Revenue Distribution', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('campaign_revenue.png', dpi=150)
    plt.show()
    print('✅ Chart saved: campaign_revenue.png')

def plot_category_performance(category_df):
    """Bar chart — Revenue by product category."""
    pdf = category_df.orderBy('category_revenue', ascending=False).toPandas()
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(pdf['category'], pdf['category_revenue'], color='#2ca02c', edgecolor='white')
    ax.set_title('Revenue by Product Category', fontsize=14, fontweight='bold')
    ax.set_xlabel('Category')
    ax.set_ylabel('Total Revenue ($)')
    ax.bar_label(bars, fmt='$%.0f', padding=3, fontsize=9)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig('category_performance.png', dpi=150)
    plt.show()
    print('✅ Chart saved: category_performance.png')

# Generate all charts
print('📊 Generating Business Intelligence Charts...')
plot_top_customers(cust_df)
plot_top_products(prod_df)
plot_campaign_revenue(camp_df)
plot_category_performance(cat_df)

---
## 💾 Step 6 — Delta Lake Storage (delta_utils.py)

Persists all analytics results as **Delta Lake tables** with:
- ACID transaction guarantees
- ZORDER optimization for fast queries
- VACUUM for storage cleanup

In [ ]:
# ============================================================
# delta_utils.py — Delta Lake Operations
# ============================================================

def create_delta_table(df, table_name, db='analytics_db'):
    """Write DataFrame as Delta table in the analytics database."""
    spark = SparkSession.getActiveSession()
    spark.sql(f'CREATE DATABASE IF NOT EXISTS {db}')
    df.write.format('delta').mode('overwrite').saveAsTable(f'{db}.{table_name}')
    row_count = spark.table(f'{db}.{table_name}').count()
    print(f'   ✅ {db}.{table_name} — {row_count:,} rows saved')

def optimize_table(table):
    """OPTIMIZE with ZORDER and VACUUM Delta table."""
    spark = SparkSession.getActiveSession()
    spark.sql(f'OPTIMIZE {table} ZORDER BY (customer_id)')
    spark.sql(f'VACUUM {table} RETAIN 168 HOURS')
    print(f'   ✅ Optimized: {table}')

def create_data_quality_table(df, db='analytics_db'):
    """Create data quality summary Delta table."""
    null_summary = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
    null_summary = null_summary.withColumn('total_rows', lit(df.count()))
    create_delta_table(null_summary, 'data_quality_summary', db)

# Save all analytics tables
print('💾 Saving to Delta Lake...')
print('=' * 50)
tables_to_save = [
    (joined,   'transactions_insights'),
    (cust_df,  'customer_analytics'),
    (prod_df,  'product_analytics'),
    (cat_df,   'category_analytics'),
    (camp_df,  'campaign_analytics'),
]
for df, name in tables_to_save:
    create_delta_table(df, name)
create_data_quality_table(tx_df)

# Optimize all tables
print('\n⚡ Optimizing Delta tables...')
for _, name in tables_to_save:
    optimize_table(f'analytics_db.{name}')

print('\n🎉 All tables saved and optimized!')

---
## 🔍 Step 7 — Query Analytics Results

Sample SQL queries on the Delta tables.

In [ ]:
# ============================================================
# Business Intelligence Queries on Delta Tables
# ============================================================

print('📊 Top 10 Customers by Revenue:')
spark.sql("""
    SELECT customer_id, total_transactions, total_revenue, avg_order_value, is_revenue_outlier
    FROM analytics_db.customer_analytics
    ORDER BY total_revenue DESC
    LIMIT 10
""").show()

print('📣 Campaign Performance Dashboard:')
spark.sql("""
    SELECT campaign_id, campaign_revenue, total_transactions,
           unique_customers, avg_revenue_per_customer
    FROM analytics_db.campaign_analytics
    ORDER BY campaign_revenue DESC
""").show()

print('🏷️  Category Revenue Ranking:')
spark.sql("""
    SELECT category, units_sold, category_revenue
    FROM analytics_db.category_analytics
    ORDER BY category_revenue DESC
""").show()

print('🛡️  Data Quality Summary:')
spark.sql("""
    SELECT * FROM analytics_db.data_quality_summary
""").show()

---
## 🎯 Step 8 — Pipeline Summary

In [ ]:
print('=' * 60)
print('  🎉 AZURE TRANSACTION ANALYTICS PIPELINE — COMPLETE')
print('=' * 60)
print()
print('  Pipeline Stages Completed:')
print('  ✅ 1. ADLS Gen2 connection & file verification')
print('  ✅ 2. Schema-enforced data ingestion (transactions + products)')
print('  ✅ 3. Data cleaning — null removal, dedup, outlier detection')
print('  ✅ 4. Quality flags — score, timestamp, record hash')
print('  ✅ 5. Channel enrichment (Web / Mobile / In-Store)')
print('  ✅ 6. Analytics — Customer LTV, Product, Category, Campaign')
print('  ✅ 7. Business charts generated (4 visualizations)')
print('  ✅ 8. Delta Lake tables created and optimized')
print()
print('  Delta Tables in analytics_db:')
spark.sql('SHOW TABLES IN analytics_db').show()
print()
print('  Developer : Sangam Srivastav')
print('  Internship: Celebal Technologies — Data Engineering')
print('  Stack     : PySpark · Azure Databricks · ADLS Gen2 · Delta Lake')
print('=' * 60)